# GuitarMidi-LV2 Library
 # Copyright (C) 2026 Gerald Mwangi
 #
 # This program is free software; you can redistribute it and/or
 # modify it under the terms of the GNU Lesser General Public
 # License as published by the Free Software Foundation; either
 # version 2 of the License, or (at your option) any later version.
 #
 # This program is distributed in the hope that it will be useful,
 # but WITHOUT ANY WARRANTY; without even the implied warranty of
 # MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the GNU
 # Lesser General Public License for more details.
 #
 # You should have received a copy of the GNU Lesser General
 # Public License along with this program; if not, write to the
 # Free Software Foundation, Inc., 51 Franklin Street, Fifth Floor,
 # Boston, MA  02110-1301  USA

In [1]:
import tensorflow as tf
import os
import glob,re
import random
import numpy as np
from model import build_1d_cnn_model
from common import INPUT_SHAPE,OUTPUT_DIM_NOTES, fast_gpu_map,parse_filtered_audio_record
cnn_model=build_1d_cnn_model(batch_sz=1,input_shape=INPUT_SHAPE,output_dim=37,training=False,with_gru=True)
cnn_model.summary()
#tf.keras.utils.plot_model(cnn_model,to_file='cnn_model.png',show_shapes=True)
cnn_model.load_weights('/home/gerald/workspace/src/GuitarMidi-LV2-release/python/neuralnetmodelling/checkpoints/guitarmidi_staticmapping_gru_epoch247_valAcc0.9881_valPrec0.8365_valRecall0.7354.keras')#
#cnn_model.load_weights('guitarmidi.keras')


input_filepaths = '/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/training_subset/training_subset_electric'#sorted(glob.glob(os.path.join(input_data_dir, '**', 'input', 'data.tfrecord'), recursive=True))
input_filepaths=glob.glob(os.path.join(input_filepaths, '**', '*.tfrecord'), recursive=True)
input_filepaths = sorted(input_filepaths,key=lambda file: int(re.findall('\\d+',os.path.basename(file))[0]) )
random.shuffle(input_filepaths)
# train_dataset = tf.data.Dataset.from_tensor_slices((input_filepaths))
# train_dataset=train_dataset.shuffle(buffer_size=len(input_filepaths))
# train_dataset=train_dataset.take(100)
# train_dataset = train_dataset.map(tf_load_sample_from_files, num_parallel_calls=tf.data.AUTOTUNE)
def representative_data_gen():
    # Use TFRecordDataset to actually read the files
    # We only need a few samples to calibrate quantization
    raw_dataset = tf.data.TFRecordDataset(input_filepaths).map(parse_filtered_audio_record).shuffle(buffer_size=len(input_filepaths)).take(1000)
    
    # Map using your existing loading function
    calib_dataset = raw_dataset.map(lambda it,ot: fast_gpu_map(it,ot, training=False))
    
    for input_value, _ in calib_dataset.batch(1):
        # input_value is the (312, 256, 1) tensor
        yield [input_value]

converter = tf.lite.TFLiteConverter.from_keras_model(cnn_model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]
# converter.representative_dataset = representative_data_gen  
# converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
#converter._experimental_lower_tensor_list_ops = True 
# converter.inference_input_type = tf.int8
# converter.inference_output_type = tf.int8
tflite_model = converter.convert()


# converter=tf.lite.TFLiteConverter.from_keras_model(cnn_model)
# converter.optimizations = [tf.lite.Optimize.DEFAULT]
# tflite_model=converter.convert()

with open('guitarmidi.tflite','wb') as f:
    f.write(tflite_model)
print("TFLite model saved as guitarmidi.tflite")

I0000 00:00:1782553140.475497  677231 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782553140.493265  677231 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1782553141.532727  677231 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Image height:  148
Before string split: (1, 148, 64), max_x=148.0
String slice ranges (in time steps):  [(0, 52), (20, 72), (40, 92), (60, 112), (76, 128), (96, 148)]
String 0: slicing from 0 to 52 (max_x=148.0)
String 1: slicing from 20 to 72 (max_x=148.0)
String 2: slicing from 40 to 92 (max_x=148.0)
String 3: slicing from 60 to 112 (max_x=148.0)
String 4: slicing from 76 to 128 (max_x=148.0)
String 5: slicing from 96 to 148 (max_x=148.0)
After string split:  [(1, 13, 64), (1, 13, 64), (1, 13, 64), (1, 13, 64), (1, 13, 64), (1, 13, 64)]


Model: "guitar_note_detector"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_spectrogram   │ (1, 148, 256, 1)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ local_mean          │ (1, 148, 256, 1)  │          0 │ input_spectrogra… │
│ (AveragePooling2D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ local_contrast      │ (1, 148, 256, 1)  │          0 │ input_spectrogra… │
│ (Subtract)          │                   │            │ local_mean[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_to_2d       │ (1, 148, 256, 1)  │          0 │ local_contrast[0… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_compress_conv… │ (1, 148, 64, 8)   │        136 │ reshape_to_2d[0]… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_compress_bn    │ (1, 148, 64, 8)   │         32 │ freq_compress_co… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_compress_act   │ (1, 148, 64, 8)   │          0 │ freq_compress_bn… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_compress_drop  │ (1, 148, 64, 8)   │          0 │ freq_compress_ac… │
│ (SpatialDropout2D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_to_1d       │ (1, 148, 512)     │          0 │ freq_compress_dr… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ context_squeeze     │ (1, 148, 64)      │     32,832 │ reshape_to_1d[0]… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ context_squeeze_bn  │ (1, 148, 64)      │        256 │ context_squeeze[… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ context_squeeze_act │ (1, 148, 64)      │          0 │ context_squeeze_… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ harmonic_ctx_conv_… │ (1, 148, 64)      │     12,352 │ context_squeeze_… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ harmonic_ctx_bn_d1  │ (1, 148, 64)      │        256 │ harmonic_ctx_con… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ harmonic_ctx_act_d1 │ (1, 148, 64)      │          0 │ harmonic_ctx_bn_… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ harmonic_ctx_drop_… │ (1, 148, 64)      │          0 │ harmonic_ctx_act… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ harmonic_ctx_add_d1 │ (1, 148, 64)      │          0 │ context_squeeze_

 Total params: 565,082 (2.16 MB)

...

 Non-trainable params: 6,800 (26.56 KB)

INFO:tensorflow:Assets written to: /tmp/tmpnxbrf82l/assets


INFO:tensorflow:Assets written to: /tmp/tmpnxbrf82l/assets


Saved artifact at '/tmp/tmpnxbrf82l'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 148, 256, 1), dtype=tf.float32, name='input_spectrogram')
Output Type:
  TensorSpec(shape=(1, 37), dtype=tf.float32, name=None)
Captures:
  127386198619344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127386198618768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127386198621840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127386198619728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127386198621456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127386198621648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127386192790160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127386192791504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127386192792464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127386192792656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  12738619279073

W0000 00:00:1782553146.181123  677231 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1782553146.181139  677231 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1782553146.181430  677231 reader.cc:83] Reading SavedModel from: /tmp/tmpnxbrf82l
I0000 00:00:1782553146.186692  677231 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1782553146.186701  677231 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpnxbrf82l
I0000 00:00:1782553146.235432  677231 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1782553146.244314  677231 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1782553146.564158  677231 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpnxbrf82l
I0000 00:00:1782553146.650272  677231 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 468851 microseconds.
I0000 00:00:1782553146.766853  67723